In [ ]:
!pip install spacy PyPDF2 pandas wordcloud matplotlib scikit-learn sentence-transformers nltk --quiet
!python -m spacy download en_core_web_md --quiet

import spacy, re, os, PyPDF2, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from wordcloud import WordCloud
from sentence_transformers import SentenceTransformer
from google.colab import files
from collections import Counter
from IPython.display import display, HTML
import nltk, warnings

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)

uploaded = files.upload()
os.makedirs("resumes", exist_ok=True)
for fname in uploaded:
    dest = os.path.join("resumes", fname)
    if os.path.exists(dest):
        os.remove(dest)
    os.rename(fname, dest)

nlp = spacy.load("en_core_web_md")
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")


def read_pdf(path):
    text = ""
    try:
        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                content = page.extract_text()
                if content:
                    text += content + "\n"
    except:
        pass
    return text.strip()


def get_name(doc):
    for ent in doc.ents:
        if ent.label_ == "PERSON" and len(ent.text.split()) >= 2:
            return ent.text.strip()
    for line in doc.text.strip().split("\n")[:5]:
        line = line.strip()
        if 2 <= len(line.split()) <= 5 and line.replace(" ", "").isalpha():
            return line
    return "Not Found"


def get_email(text):
    match = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-z]{2,}", text)
    return match[0] if match else "Not Found"


def get_phone(text):
    matches = re.findall(r"(\+?\d[\d\s\-\(\)]{8,14}\d)", text)
    for m in matches:
        if 10 <= len(re.sub(r"\D", "", m)) <= 13:
            return m.strip()
    return "Not Found"


def get_linkedin(text):
    match = re.findall(r"linkedin\.com/in/[\w\-]+", text, re.IGNORECASE)
    return "https://" + match[0] if match else "Not Found"


def get_github(text):
    match = re.findall(r"github\.com/[\w\-]+", text, re.IGNORECASE)
    return "https://" + match[0] if match else "Not Found"


def get_education(doc):
    keywords = ["b.tech","b.e","m.tech","bachelor","master","b.sc","m.sc",
                "bca","mca","phd","diploma","cgpa","gpa","10th","12th"]
    lines = [s.text.strip() for s in doc.sents if any(k in s.text.lower() for k in keywords)]
    return " | ".join(lines[:3]) if lines else "Not Found"


def get_experience(doc):
    keywords = ["experience","intern","project","developer","engineer","analyst","worked","trainee"]
    lines = [s.text.strip() for s in doc.sents if any(k in s.text.lower() for k in keywords)]
    return " | ".join(lines[:3]) if lines else "Fresher"


def get_orgs(doc):
    orgs = list(dict.fromkeys([e.text.strip() for e in doc.ents if e.label_ == "ORG"]))
    return ", ".join(orgs[:4]) if orgs else "Not Found"


def get_exp_years(text):
    match = re.findall(r"(\d+)\+?\s*year[s]?\s*(of)?\s*experience", text, re.IGNORECASE)
    if match:
        return f"{match[0][0]} year(s)"
    return "Fresher" if any(w in text.lower() for w in ["fresher","entry level"]) else "Not Specified"


SKILLS_DB = {
    "Programming":      ["Python","Java","C++","C","R","Scala","Go","Kotlin","Swift"],
    "Web":              ["HTML","CSS","JavaScript","React","Angular","Vue","Node","Django","Flask","FastAPI"],
    "Data Science/ML":  ["Machine Learning","Deep Learning","NLP","Computer Vision","Statistics",
                         "Pandas","NumPy","Scikit-learn","TensorFlow","PyTorch","Keras",
                         "spaCy","NLTK","XGBoost","LightGBM","Hugging Face"],
    "Databases":        ["SQL","MySQL","PostgreSQL","MongoDB","SQLite","Firebase","Oracle","Redis"],
    "Cloud & DevOps":   ["AWS","Azure","GCP","Docker","Kubernetes","Git","GitHub","Linux"],
    "Visualization":    ["Tableau","Power BI","Excel","Matplotlib","Seaborn","Plotly"],
    "Tools":            ["Jupyter","VS Code","Postman","JIRA","Figma"]
}

ALL_SKILLS = [s for group in SKILLS_DB.values() for s in group]


def get_skills(text):
    t = text.lower()
    return [s for s in ALL_SKILLS if s.lower() in t]


def categorize_skills(found):
    return {cat: [s for s in skills if s in found]
            for cat, skills in SKILLS_DB.items()
            if any(s in found for s in skills)}


DOMAIN_PROFILES = {
    "Data Science":      "machine learning pandas numpy statistics data analysis prediction",
    "AI/ML Engineer":    "deep learning neural network nlp computer vision transformer pytorch tensorflow",
    "Web Developer":     "html css javascript react node express frontend backend api",
    "Software Engineer": "java c++ algorithms oop data structures system design",
    "Data Analyst":      "excel tableau power bi sql visualization dashboard reporting",
    "Cloud Engineer":    "aws azure gcp docker kubernetes devops ci/cd",
    "Mobile Dev":        "android kotlin swift ios flutter react native"
}

tfidf = TfidfVectorizer()
domain_vecs = tfidf.fit_transform(DOMAIN_PROFILES.values())


def classify_domain(text, skills):
    vec = tfidf.transform([text + " " + " ".join(skills)])
    sims = cosine_similarity(vec, domain_vecs).flatten()
    labels = list(DOMAIN_PROFILES.keys())
    top = np.argsort(sims)[::-1][:2]
    scores = {labels[i]: round(float(sims[i]) * 100, 1) for i in top if sims[i] > 0}
    return labels[top[0]] if sims[top[0]] > 0 else "General", scores


def score_resume(data):
    s = {}
    s["Contact"]  = sum([data["Email"] != "Not Found",
                         data["Phone"] != "Not Found",
                         data["LinkedIn"] != "Not Found",
                         data["GitHub"] != "Not Found"]) * 5
    s["Skills"]   = min(len(data["skills_list"]) * 2, 35)
    s["Education"]= 20 if data["Education"] != "Not Found" else 0
    s["Experience"]= 0 if data["Experience"] == "Fresher" else 15
    s["Presence"] = (5 if data["GitHub"] != "Not Found" else 0) + \
                    (5 if data["LinkedIn"] != "Not Found" else 0)

    total = min(sum(s.values()), 100)
    grade = "Excellent" if total >= 80 else "Good" if total >= 60 else "Average" if total >= 40 else "Needs Work"
    return total, grade, s


def match_with_jd(resume_text, jd_text, resume_skills):
    emb = semantic_model.encode([resume_text[:1000], jd_text])
    sem = float(cosine_similarity([emb[0]], [emb[1]])[0][0]) * 100
    jd_skills = [s for s in ALL_SKILLS if s.lower() in jd_text.lower()]
    matched = [s for s in resume_skills if s in jd_skills]
    missing = [s for s in jd_skills if s not in resume_skills]
    skill_pct = (len(matched) / len(jd_skills) * 100) if jd_skills else 0
    return {
        "overall":  round((sem * 0.5 + skill_pct * 0.5), 1),
        "semantic": round(sem, 1),
        "skills":   round(skill_pct, 1),
        "matched":  matched,
        "missing":  missing
    }


def parse(path):
    text = read_pdf(path)
    if not text:
        return None
    doc = nlp(text)
    skills = get_skills(text)
    domain, domain_scores = classify_domain(text, skills)

    data = {
        "File":         os.path.basename(path),
        "Name":         get_name(doc),
        "Email":        get_email(text),
        "Phone":        get_phone(text),
        "LinkedIn":     get_linkedin(text),
        "GitHub":       get_github(text),
        "Education":    get_education(doc),
        "Experience":   get_experience(doc),
        "Exp_Years":    get_exp_years(text),
        "Orgs":         get_orgs(doc),
        "skills_list":  skills,
        "Skills":       ", ".join(skills) if skills else "Not Found",
        "Skill_Groups": categorize_skills(skills),
        "Skill_Count":  len(skills),
        "Domain":       domain,
        "Domain_Scores":domain_scores,
    }
    score, grade, breakdown = score_resume(data)
    data["Score"]     = score
    data["Grade"]     = grade
    data["Breakdown"] = breakdown
    return data


results = []
for f in os.listdir("resumes"):
    if f.lower().endswith(".pdf"):
        r = parse(os.path.join("resumes", f))
        if r:
            results.append(r)

cols = ["File","Name","Email","Phone","LinkedIn","GitHub",
        "Education","Exp_Years","Skills","Skill_Count","Domain","Score","Grade"]
df = pd.DataFrame(results)[cols]

display(HTML("""<style>
table{border-collapse:collapse;font-family:Arial;font-size:13px;width:100%}
th{background:#4A90D9;color:white;padding:8px 12px;text-align:left}
td{padding:7px 12px;border-bottom:1px solid #ddd}
tr:hover{background:#f0f7ff}
</style>"""))
display(HTML(df.to_html(index=False)))

for r in results:
    print(f"\n{'─'*50}")
    print(f"  {r['File']}")
    print(f"{'─'*50}")
    print(f"  Name       : {r['Name']}")
    print(f"  Email      : {r['Email']}")
    print(f"  Phone      : {r['Phone']}")
    print(f"  LinkedIn   : {r['LinkedIn']}")
    print(f"  GitHub     : {r['GitHub']}")
    print(f"  Education  : {r['Education'][:80]}")
    print(f"  Experience : {r['Experience'][:80]}")
    print(f"  Orgs       : {r['Orgs']}")
    print(f"\n  Skills by Category:")
    for cat, sk in r["Skill_Groups"].items():
        print(f"    {cat:20s} -> {', '.join(sk)}")
    print(f"\n  Domain     : {r['Domain']}  {r['Domain_Scores']}")
    print(f"  Score      : {r['Score']}/100  ({r['Grade']})")
    print(f"  Breakdown  : {r['Breakdown']}")


fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Resume Analysis Dashboard", fontsize=17, fontweight="bold")

all_skills_flat = [s for r in results for s in r["skills_list"]]

ax1 = axes[0, 0]
if all_skills_flat:
    wc = WordCloud(width=700, height=350, background_color="white",
                   colormap="Blues", max_words=60).generate(" ".join(all_skills_flat))
    ax1.imshow(wc, interpolation="bilinear")
ax1.axis("off")
ax1.set_title("Skills Word Cloud", fontsize=13, fontweight="bold")

ax2 = axes[0, 1]
domain_counts = Counter([r["Domain"] for r in results])
colors = plt.cm.Set3(np.linspace(0, 1, len(domain_counts)))
bars = ax2.bar(domain_counts.keys(), domain_counts.values(), color=colors, edgecolor="white")
ax2.set_title("Domain Distribution", fontsize=13, fontweight="bold")
ax2.tick_params(axis='x', rotation=30)
for b in bars:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
             str(int(b.get_height())), ha="center", fontsize=10, fontweight="bold")

ax3 = axes[1, 0]
names  = [r["Name"].split()[0] if r["Name"] != "Not Found" else r["File"][:10] for r in results]
scores = [r["Score"] for r in results]
bar_colors = ["#2ecc71" if s >= 80 else "#f39c12" if s >= 60 else "#e74c3c" for s in scores]
bars3 = ax3.barh(names, scores, color=bar_colors, edgecolor="white")
ax3.set_xlim(0, 110)
ax3.set_title("Resume Strength Scores", fontsize=13, fontweight="bold")
for b, sc in zip(bars3, scores):
    ax3.text(b.get_width() + 1, b.get_y() + b.get_height()/2,
             f"{sc}/100", va="center", fontsize=10, fontweight="bold")
ax3.legend(handles=[
    mpatches.Patch(color="#2ecc71", label="Excellent (80+)"),
    mpatches.Patch(color="#f39c12", label="Good (60-79)"),
    mpatches.Patch(color="#e74c3c", label="Needs Work (<60)")
], loc="lower right", fontsize=8)

ax4 = axes[1, 1]
top_skills = Counter(all_skills_flat).most_common(10)
if top_skills:
    sk_names, sk_vals = zip(*top_skills)
    cols4 = plt.cm.viridis(np.linspace(0.3, 0.9, len(sk_names)))
    bars4 = ax4.bar(sk_names, sk_vals, color=cols4, edgecolor="white")
    ax4.set_title("Top 10 Skills Frequency", fontsize=13, fontweight="bold")
    ax4.tick_params(axis='x', rotation=40)
    for b in bars4:
        ax4.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
                 str(int(b.get_height())), ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("dashboard.png", dpi=150, bbox_inches="tight")
plt.show()


print("\nPaste a Job Description and press Enter twice when done. Type 'skip' to skip.\n")
jd_lines = []
while True:
    line = input()
    if line.lower() == "skip":
        jd_lines = []
        break
    if line == "" and jd_lines and jd_lines[-1] == "":
        break
    jd_lines.append(line)

jd_text = "\n".join(jd_lines).strip()
if jd_text:
    for r in results:
        text = read_pdf(os.path.join("resumes", r["File"]))
        m = match_with_jd(text, jd_text, r["skills_list"])
        print(f"\n  {r['Name']} — {r['File']}")
        print(f"    Overall Match : {m['overall']}%")
        print(f"    Semantic      : {m['semantic']}%  |  Skill Match: {m['skills']}%")
        print(f"    Matched       : {', '.join(m['matched']) or 'None'}")
        print(f"    Missing       : {', '.join(m['missing'][:6]) or 'None'}")


export = pd.DataFrame([{
    "File": r["File"], "Name": r["Name"], "Email": r["Email"],
    "Phone": r["Phone"], "LinkedIn": r["LinkedIn"], "GitHub": r["GitHub"],
    "Education": r["Education"], "Experience": r["Experience"],
    "Exp_Years": r["Exp_Years"], "Orgs": r["Orgs"],
    "Skills": r["Skills"], "Skill_Count": r["Skill_Count"],
    "Domain": r["Domain"], "Score": r["Score"], "Grade": r["Grade"]
} for r in results])

export.to_csv("results.csv", index=False)
files.download("results.csv")
files.download("dashboard.png")